<a href="https://colab.research.google.com/github/blskyue/AI-Research-Agent/blob/main/ai_research_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
from google.colab import userdata
import os

# هذي الطريقة الصحيحة لسحب المفتاح من Secrets
os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')
os.environ["TAVILY_API_KEY"] = userdata.get('TAVILY_API_KEY')

print("تم تفعيل المفاتيح بنجاح!")

تم تفعيل المفاتيح بنجاح!


In [8]:
!pip install -U langchain-openai langgraph langchain-community tavily-python

In [9]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain_community.tools.tavily_search import TavilySearchResults
import requests
from langchain.tools import tool

# 1. تعريف الأدوات المطلوبة في الواجب
internet_search = TavilySearchResults(k=3) # أداة البحث

@tool
def fetch_url(url: str) -> str:
    """Fetch text content from a URL"""
    try:
        response = requests.get(url, timeout=10.0)
        response.raise_for_status()
        return response.text
    except Exception as e:
        return f"Error fetching URL: {e}"

tools = [internet_search, fetch_url]

# 2. إعداد الموديل (Nemotron)
model_nemotron = ChatOpenAI(
    model="google/gemma-4-31b-it:free",
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    openai_api_base="https://openrouter.ai/api/v1"
)

# 3. بناء الوكيل مع التعليمات (System Prompt)
system_prompt = "أنت مساعد بحث ذكي. ابحث عن الموضوع، ثم استخدم fetch_url لقراءة أول 3 نتائج بتمعن، ثم لخص الإجابة النهائية بناءً على ما قرأت."
# بدال state_modifier حطينا prompt
agent_executor = create_react_agent(model_nemotron, tools, prompt=system_prompt)

print("✅ تم بناء الوكيل بنجاح! جاهز للاختبار.")

✅ تم بناء الوكيل بنجاح! جاهز للاختبار.


/tmp/ipykernel_22992/1810993675.py:32: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(model_nemotron, tools, prompt=system_prompt)


In [10]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

# استخدمنا موديل Gemma 2 المطور والمجاني
model_gemma = ChatOpenAI(
    model="google/gemma-2-9b-it",
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    openai_api_base="https://openrouter.ai/api/v1"
)

# بناء الوكيل من جديد بالموديل الجديد
agent_executor = create_react_agent(model_gemma, tools, prompt=system_prompt)

print("✅ تم تحديث الموديل إلى Gemma بنجاح!")

✅ تم تحديث الموديل إلى Gemma بنجاح!


/tmp/ipykernel_22992/1207917732.py:12: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(model_gemma, tools, prompt=system_prompt)


In [11]:
import requests
import json

# OpenRouter API base URL
OPENROUTER_API_BASE = "https://openrouter.ai/api/v1"

# Endpoint to list models (this is a common pattern, might need adjustment if OpenRouter uses a different one)
models_url = f"{OPENROUTER_API_BASE}/models"

headers = {
    "Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}",
    "Content-Type": "application/json"
}

print("Attempting to fetch available models from OpenRouter...")

try:
    response = requests.get(models_url, headers=headers)
    response.raise_for_status() # Raise an exception for HTTP errors
    models_data = response.json()

    print("\nAvailable Models on OpenRouter:")
    # Print only the model IDs for clarity
    for model in models_data.get('data', []):
        print(model.get('id'))

except requests.exceptions.RequestException as e:
    print(f"Error fetching models: {e}")
    if response.status_code == 401:
        print("Please ensure your OPENROUTER_API_KEY is correct and active.")
    elif response.status_code == 404:
        print("The models endpoint might be different or unavailable. Please check OpenRouter's API documentation.")
except json.JSONDecodeError:
    print("Error decoding JSON response from OpenRouter.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Attempting to fetch available models from OpenRouter...

Available Models on OpenRouter:
anthropic/claude-opus-4.6-fast
z-ai/glm-5.1
google/gemma-4-26b-a4b-it:free
google/gemma-4-26b-a4b-it
google/gemma-4-31b-it:free
google/gemma-4-31b-it
qwen/qwen3.6-plus
z-ai/glm-5v-turbo
arcee-ai/trinity-large-thinking
x-ai/grok-4.20-multi-agent
x-ai/grok-4.20
google/lyria-3-pro-preview
google/lyria-3-clip-preview
kwaipilot/kat-coder-pro-v2
rekaai/reka-edge
xiaomi/mimo-v2-omni
xiaomi/mimo-v2-pro
minimax/minimax-m2.7
openai/gpt-5.4-nano
openai/gpt-5.4-mini
mistralai/mistral-small-2603
z-ai/glm-5-turbo
nvidia/nemotron-3-super-120b-a12b:free
nvidia/nemotron-3-super-120b-a12b
bytedance-seed/seed-2.0-lite
qwen/qwen3.5-9b
openai/gpt-5.4-pro
openai/gpt-5.4
inception/mercury-2
openai/gpt-5.3-chat
google/gemini-3.1-flash-lite-preview
bytedance-seed/seed-2.0-mini
google/gemini-3.1-flash-image-preview
qwen/qwen3.5-35b-a3b
qwen/qwen3.5-27b
qwen/qwen3.5-122b-a10b
qwen/qwen3.5-flash-02-23
liquid/lfm-2-24b-a2b
goo

In [13]:
# هذي الخلية لتشغيل الوكيل وسؤاله
query = "ما هي أبرز مشاريع سدايا في الذكاء الاصطناعي لعام 2025؟"

print(f"جاري البحث عن: {query}...\n")

# تشغيل الوكيل ورؤية النتائج
for chunk in agent_executor.stream({"messages": [("user", query)]}):
    if "messages" in chunk:
        # طباعة رد الوكيل النهائي أو خطواته
        for message in chunk["messages"]:
            if hasattr(message, 'content') and message.content:
                print(message.content)
                print("-" * 50)

جاري البحث عن: ما هي أبرز مشاريع سدايا في الذكاء الاصطناعي لعام 2025؟...



NotFoundError: Error code: 404 - {'error': {'message': 'No endpoints found that support tool use. Try disabling "tavily_search_results_json". To learn more about provider routing, visit: https://openrouter.ai/docs/guides/routing/provider-selection', 'code': 404}}